# AI Hiring Bias Detector & Algorithmic Fairness Audit Framework

**Dataset:** 250 Candidate Screening Records across 5 Technical Job Roles and 4 Geographic Regions in Indonesia.  
**Objective:** Pendeteksian bias algoritma AI pada proses rekrutmen (*Algorithmic Bias Audit*), pengujian metrik Keadilan Algoritma (*Demographic Parity & Disparate Impact Ratio*), serta penyesuaian penalti bias gender dan geografis.

### Stage Breakdown:
1. **Data Preprocessing & Audit Setup**: Pemuatan data pelamar kerja dan verifikasi variabel kualifikasi baseline.
2. **Fairness Metric Formulation**: Perhitungan *Disparate Impact Ratio (DIR)* dan *Demographic Parity Difference*.
3. **Publication-Grade Visual Analysis**: Visualisasi terpisah 300 DPI (DIR per Peran Kerja, Distribusi Skor Geografis, Matriks Heatmap Keadilan, dan Scatter Penalti Kualifikasi).
4. **Econometric Bias Modeling**: Model OLS regresi penentu penalti skor seleksi AI.
5. **Algorithmic Remediation**: Pengujian ambang batas keadilan EEOC 80% rule.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.formula.api import ols

os.makedirs('images', exist_ok=True)
os.makedirs('data', exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.titlesize': 14,
    'figure.dpi': 300,
    'savefig.dpi': 300
})

print('Environment setup complete. Output directory ready.')

In [ ]:
df = pd.read_csv('data/hiring_bias_dataset.csv')
print('--- Dataset Shape ---')
print(df.shape)
print('\n--- Head Data ---')
print(df.head())

In [ ]:
# Regresi OLS Penentu Skor Seleksi Akhir AI
model = ols('final_screening_score ~ baseline_qualification_score + gender_bias_penalty + location_bias_penalty', data=df).fit()
print(model.summary())

In [ ]:
# 1. Disparate Impact Ratio by Job Role & Gender Wording Cues
fig, ax = plt.subplots(figsize=(8, 5))
role_gender = df.groupby(['job_role', 'gender_text_cue'])['disparate_impact_ratio'].mean().unstack()
role_gender.plot(kind='bar', ax=ax, colormap='Set1', width=0.7, edgecolor='black', linewidth=0.5)
ax.set_title('Disparate Impact Ratio by Job Role & Gender Wording Cues', fontweight='bold', pad=12)
ax.set_xlabel('Job Role')
ax.set_ylabel('Mean Disparate Impact Ratio (DIR)')
ax.axhline(0.80, color='#b91c1c', linestyle='--', label='80% EEOC Fairness Threshold')
ax.legend(loc='lower right', frameon=True, title='Gender Cue')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('images/disparate_impact_by_role_gender.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Distribution of AI Screening Scores Across Candidate Locations
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='candidate_location', y='final_screening_score', hue='candidate_location', palette='Blues', ax=ax, width=0.5, legend=False)
ax.set_title('Distribution of AI Screening Scores Across Candidate Locations', fontweight='bold', pad=12)
ax.set_xlabel('Candidate Geographic Location')
ax.set_ylabel('AI Final Screening Score (0-100)')
plt.tight_layout()
plt.savefig('images/geographic_screening_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Fairness Heatmap Matrix: Disparate Impact Ratio (Role vs Location)
fig, ax = plt.subplots(figsize=(8, 6))
pivot_heatmap = df.pivot_table(index='job_role', columns='candidate_location', values='disparate_impact_ratio', aggfunc='mean')
sns.heatmap(pivot_heatmap, annot=True, fmt='.3f', cmap='YlOrRd_r', ax=ax, cbar=True, linewidths=0.5)
ax.set_title('Fairness Heatmap Matrix: Disparate Impact Ratio (Role vs Location)', fontweight='bold', pad=12)
ax.set_xlabel('Candidate Location')
ax.set_ylabel('Job Role')
plt.tight_layout()
plt.savefig('images/fairness_heatmap_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Baseline Qualification vs Final AI Screening Score
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x='baseline_qualification_score',
    y='final_screening_score',
    hue='gender_text_cue',
    style='candidate_location',
    s=80,
    alpha=0.8,
    ax=ax,
    palette='Set2'
)
ax.plot([30, 100], [30, 100], color='black', linestyle=':', label='Parity Line (No Bias)')
ax.set_title('Baseline Qualification vs Final AI Screening Score', fontweight='bold', pad=12)
ax.set_xlabel('Baseline Qualification Score')
ax.set_ylabel('Final AI Screening Score')
ax.legend(loc='upper left', frameon=True, bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig('images/qualification_vs_final_screening_score.png', dpi=300, bbox_inches='tight')
plt.show()